<h2>Import Libraries<h2>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import RandomOverSampler


<h2>Import Dataset<h2>

In [ ]:
df_loan=pd.read_csv('../data/cleaned_data/cleaned_loan_dataset.csv')
df_loan

<h2>Separate Features and Target</h2>

<p>Drop the loan_id (identifier, not a predictive feature) and isolate the target variable loan_status.</p>

In [ ]:
X = df_loan.drop(columns=['loan_id', 'loan_status',  'total_income', 'balance_income'])
y = df_loan['loan_status']

In [ ]:
print('Features shape:', X.shape)
print('Target shape:', y.shape)

<h2>Split into Train and Test Sets</h2>

<p>Use a stratified split to preserve the loan_status class distribution in both sets. The split happens before any scaling/encoding is fit
to avoid data leakage.</p>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
print('Train features shape:', X_train.shape)
print('Test features shape :', X_test.shape)

<h2>Scale Numeric Features</h2>
<p>Scale only the numeric columns using StandardScaler. Fit the scaler on the training set only (to avoid data leakage) and apply it to both train and test. </p>

In [ ]:
# Separate numeric and categorical/encoded columns
numeric_cols = ['loan_amount', 'loan_amount_term', 'applicant_income',
                'coapplicant_income',
                'estimated_monthly_installment']
    
cat_cols = [c for c in X.columns if c not in numeric_cols]

print('Numeric columns:', numeric_cols)
print('Categorical (unscaled) columns:', cat_cols)

In [ ]:
# Scale only the numeric columns and fit on train, apply to both
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

<h3>Feature Selection</h3>

In [ ]:
# Five most statistically relevant features using training data only
selector = SelectKBest(score_func=f_classif,  k=5)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X_train_scaled.columns[selector.get_support()]

print("Selected features:")
print(list(selected_features))
print("Training shape:", X_train_selected.shape)
print("Testing shape :", X_test_selected.shape)

<h3>Helper function to evaluate the train and test accuracy</h3>

In [ ]:
# A helper function to evaluate the train and test accuracy
def evaluate_accuracy(model, X_train_selected, y_train, X_test_selected, y_test):

    # Train accuracy
    y_pred_train = model.predict(X_train_selected)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    
    # Test accuracy
    y_pred_test = model.predict(X_test_selected)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    print(f'Train Accuracy: {round((train_accuracy * 100), 2)}%')
    print(f'Test Accuracy: {round((test_accuracy * 100) ,2)}%')

    print(" ")

    # Classification Report
    print("Classification Report")
    print(classification_report(y_test, y_pred_test))

    # Confusion Matrix
    print("Confusion Matrix")
    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred_test),
        columns=['Predicted N', 'Predicted Y'], 
        index=['Actual N', 'Actual Y']
    )
    print(cm)


<h3>Logistic Regression Model</h3>

In [ ]:
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_selected, y_train)

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_ros, y_train_ros)

<h3>Random Forest Model</h3>

In [ ]:
rf_model = RandomForestClassifier(max_depth = 5, class_weight= 'balanced', n_estimators= 50, random_state=42)
rf_model.fit(X_train_selected, y_train)

<h3>Evaluate Model Performance</h3>

In [ ]:
# Logistic Regression Model Performance
evaluate_accuracy(lr_model, X_train_selected, y_train, X_test_selected, y_test)

In [ ]:
# Random Forest Model Performance
evaluate_accuracy(rf_model, X_train_selected, y_train, X_test_selected, y_test)